In [ ]:
# import dependencies
%matplotlib inline
import os
import sys
import numpy as np
import pandas as pd
import skimage.io as io
from skimage.io import imread
from skimage.measure import regionprops, label
from skimage.feature import graycomatrix, graycoprops
from skimage import img_as_ubyte
from skimage.measure import shannon_entropy
from skimage.util import view_as_windows
import matplotlib.pyplot as plt
import math
import tqdm  # For progress tracking
from scipy.stats import pearsonr

# set a working directory
wdir = ('/Users/shihongwu/pancreatic_image_analysis/34434_1/')
os.chdir(wdir)

# -------------------------------------------------------
# FEATURE CALCULATION HELPERS
# -------------------------------------------------------

In [ ]:
# --------------------------------------
# 1. Pearson Correlation + Intensity Ratio
# --------------------------------------
def calculate_pearson_and_intensity_ratio(cell_mask, image1, image2):
    """
    Calculate:
    1. Pearson’s correlation coefficient (PCC) between two channels.
    2. Mean intensity ratio (image2 / image1) to assess dominance.

    Returns:
        pcc (float): Correlation of intensity patterns.
        intensity_ratio (float): Mean(image2) / Mean(image1) in the masked region.
    """
    masked_1 = image1[cell_mask]
    masked_2 = image2[cell_mask]

    # Handle flat signals (no variance)
    if np.std(masked_1) == 0 or np.std(masked_2) == 0:
        return np.nan, np.nan

    # Pearson Correlation
    pcc, _ = pearsonr(masked_1, masked_2)

    # Intensity Ratio (e.g., NaKATPase / CK19)
    mean1 = np.mean(masked_1)
    mean2 = np.mean(masked_2)
    intensity_ratio = mean2 / mean1 if mean1 != 0 else np.nan

    return pcc, intensity_ratio

# --------------------------------------
# 2. Updated Entropy Using skimage
# --------------------------------------
def calculate_entropy_skimage(cell_mask, intensity_image):
    """
    Use skimage's built-in shannon_entropy function on the masked intensity image.
    """
    masked_intensity = intensity_image * cell_mask

    # Crop to bounding box to minimize background
    y, x = np.nonzero(cell_mask)
    cropped = masked_intensity[np.min(y):np.max(y)+1, np.min(x):np.max(x)+1]

    return shannon_entropy(cropped)

# --------------------------------------
# 3. Polarity Score
# --------------------------------------
def calculate_intensity_centroid(cell_mask, intensity_image):
    """
    Calculate intensity-weighted centroid (center of mass) of signal in the mask.
    """
    y, x = np.nonzero(cell_mask)
    values = intensity_image[cell_mask]
    cx = np.sum(x * values) / np.sum(values)
    cy = np.sum(y * values) / np.sum(values)
    return cx, cy

def calculate_polarity_score(geometric_centroid, intensity_centroid, cell_size):
    """
    Compute polarity score as Euclidean distance between geometric and intensity centroid.
    """
    distance = np.linalg.norm(np.array(geometric_centroid) - np.array(intensity_centroid))
    normalized_score = distance / (cell_size)
    return normalized_score

# --------------------------------------
# 4. Moment of Inertia (with centroid correction)
# --------------------------------------
def calculate_moment_of_inertia(cell_mask, intensity_image):
    """
    Calculate the moment of inertia of intensity around the intensity centroid.
    """
    y_coords, x_coords = np.indices(cell_mask.shape)
    cell_intensity = intensity_image * cell_mask

    # Compute intensity-weighted centroid
    total_intensity = np.sum(cell_intensity)
    if total_intensity == 0:
        return np.nan

    cx = np.sum(x_coords * cell_intensity) / total_intensity
    cy = np.sum(y_coords * cell_intensity) / total_intensity

    # Compute squared distance from centroid
    inertia = np.sum(((x_coords - cx)**2 + (y_coords - cy)**2) * cell_intensity)
    return inertia
    
# --------------------------------------
# 5. Haralick Texture Features (GLCM-based)
# --------------------------------------
def calculate_haralick_features(cell_mask, intensity_image):
    """
    Extract Haralick texture features from a masked intensity image using GLCM:
    - Contrast
    - Correlation
    - Energy
    - Homogeneity
    """
    masked_image = intensity_image * cell_mask
    masked_image = np.clip(masked_image, 0, 255).astype(np.uint8)

    # Compute Gray-Level Co-occurrence Matrix
    glcm = graycomatrix(masked_image, distances=[1], angles=[0],
                        levels=256, symmetric=True, normed=True)

    contrast = graycoprops(glcm, 'contrast')[0, 0]
    correlation = graycoprops(glcm, 'correlation')[0, 0]
    energy = graycoprops(glcm, 'energy')[0, 0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]

    return contrast, correlation, energy, homogeneity

# --------------------------------------
# 6. Lacunarity
# -----------------------------------------------------------------
def calculate_lacunarity(cell_mask, intensity_image, box_size=5):
    """
    Estimate lacunarity as a texture heterogeneity metric based on intensity variation
    within the masked region using sliding window analysis.

    Args:
        cell_mask (np.ndarray): Binary mask for the cell (same size as image).
        intensity_image (np.ndarray): Corresponding intensity image (e.g., DAPI).
        box_size (int): Size of the sliding window (odd number recommended).

    Returns:
        float: Lacunarity value (higher = more heterogeneous).
    """
    # Apply mask to intensity image
    masked_image = intensity_image * cell_mask

    # Extract bounding box to speed up
    y, x = np.nonzero(cell_mask)
    if len(y) == 0 or len(x) == 0:
        return np.nan  # avoid empty region
    cropped = masked_image[np.min(y):np.max(y)+1, np.min(x):np.max(x)+1]

    # Slide a window and compute local sums
    if cropped.shape[0] < box_size or cropped.shape[1] < box_size:
        return np.nan  # too small for window

    windows = view_as_windows(cropped, (box_size, box_size))
    local_sums = windows.sum(axis=(2, 3))

    mean = np.mean(local_sums)
    std = np.std(local_sums)

    if mean == 0:
        return np.nan  # avoid divide-by-zero

    lacunarity = (std / mean) ** 2
    return lacunarity

# -------------------------------------------------------
# MAIN PROCESSING FUNCTION
# -------------------------------------------------------

In [ ]:
def calculate_polarity_and_texture(
    cell_data, seg_dir, img_dir, fovs,
    PCC_intensity_channels,  # e.g., ["CK19", "NaKATPase"]
    intensity_channel,       # e.g., "CK19"
    dapi_channel             # e.g., "DAPI"
):
    """
    Calculate polarity, entropy, Haralick texture, and channel co-localization (PCC) per cell.

    Parameters:
        cell_data (pd.DataFrame): ARK-generated table with 'fov', 'label', 'centroid-0', 'centroid-1'.
        seg_dir (str): Path to segmentation mask directory.
        img_dir (str): Path to image directory (organized by FOV/channel).
        fovs (list): List of field of view names (strings).
        PCC_intensity_channels (list of str): Two channels to calculate PCC (e.g., ["CK19", "NaKATPase"]).
        intensity_channel (str): Channel used for polarity score.
        dapi_channel (str): Channel used for nuclear texture analysis.

    Returns:
        pd.DataFrame with extracted features for each cell.
    """
    results = []

    for fov in tqdm.tqdm(fovs, desc="Processing FOVs"):
        # Load segmentation mask
        seg_path = os.path.join(seg_dir, f"{fov}_whole_cell.tiff")
        if not os.path.exists(seg_path):
            print(f"Warning: Segmentation mask for FOV '{fov}' not found!")
            continue
        segmentation_labels = imread(seg_path).astype(int)

        # Filter cell data for current FOV
        fov_data = cell_data[cell_data['fov'] == fov]

        # Load intensity images
        polarity_img = imread(os.path.join(img_dir, f"{fov}/{intensity_channel}.tiff"))
        dapi_img = imread(os.path.join(img_dir, f"{fov}/{dapi_channel}.tiff"))
        img1 = imread(os.path.join(img_dir, f"{fov}/{PCC_intensity_channels[0]}.tiff"))
        img2 = imread(os.path.join(img_dir, f"{fov}/{PCC_intensity_channels[1]}.tiff"))

        # Loop through cells in this FOV
        for _, row in fov_data.iterrows():
            cell_label = row["label"]
            geometric_centroid = (row['centroid-1'], row['centroid-0'])  # Y, X format

            # Get binary mask for the current cell
            cell_mask = segmentation_labels == cell_label

            # Skip empty or missing masks
            if not np.any(cell_mask):
                continue

            # Use the existing cell size from cell_data
            cell_size = row['cell_size']

            # Compute features
            intensity_centroid = calculate_intensity_centroid(cell_mask, polarity_img)
            polarity_score = calculate_polarity_score(geometric_centroid, intensity_centroid, cell_size)

            inertia = calculate_moment_of_inertia(cell_mask, polarity_img)

            contrast, correlation, energy, homogeneity = calculate_haralick_features(cell_mask, dapi_img)
            
            entropy = calculate_entropy_skimage(cell_mask, dapi_img)

            lacunarity = calculate_lacunarity(cell_mask, dapi_img)
            
            pcc, intensity_ratio = calculate_pearson_and_intensity_ratio(cell_mask, img1, img2)

            # Store results
            results.append({
                "fov": fov,
                "label": cell_label,
                "geometric_centroid_x": geometric_centroid[1],
                "geometric_centroid_y": geometric_centroid[0],
                "intensity_centroid_x": intensity_centroid[0],
                "intensity_centroid_y": intensity_centroid[1],
                "polarity_score": polarity_score,
                "haralick_contrast": contrast,
                "haralick_correlation": correlation,
                "haralick_energy": energy,
                "haralick_homogeneity": homogeneity,
                "entropy": entropy,
                "pcc_ck19_nak": pcc,
                "intensity_ratio": intensity_ratio,
                "inertia": inertia,
                "lacunarity": lacunarity
            })

    return pd.DataFrame(results)

In [ ]:
# Usage
cell_data = pd.read_csv("ark_wdir/segmentation/cell_table/cell_table_arcsinh_transformed.csv")  # ARK-generated cell data
seg_dir = "ark_wdir/segmentation/deepcell_output/"
img_dir = "ark_wdir/image_data/"
fovs = cell_data['fov'].unique()  # Get unique FOVs
PCC_intensity_channels = ["CK19", "NaKATPase"]  # Specify your channels
intensity_channel = "CK19"  # Use CK19 channel for polarity calculation
dapi_channel = "DNA_1"  # Use DAPI channel for Haralick features

In [ ]:
# Calculate polarity and texture features
features_df = calculate_polarity_and_texture(cell_data, 
                                             seg_dir, 
                                             img_dir, 
                                             fovs, 
                                             PCC_intensity_channels,
                                             intensity_channel, 
                                             dapi_channel)
print(features_df.head())

In [ ]:
# Save results to a CSV
features_df.to_csv(f"results/pixel_features.csv", index=False)